<a href="https://colab.research.google.com/github/GabsCavalcant/Aplicacao-Utilizando-llm-Rn-etc/blob/main/ModeloKman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Utilizando o modelo de machine learnig não supervisionado Kmens, para criar 5 cluster (Agrupamentos) onde, consegui treinar o modelo para verificar os grupos separados de jogares, como podemos ver o 1 cluster foi um grupo mais "Valvetizado", o segundo com o enfase em toda, o terceiro jogadores mais casuais.

In [28]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("tamber/steam-video-games")

Using Colab cache for faster access to the 'steam-video-games' dataset.


In [6]:
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/tamber/steam-video-games/versions/3


In [30]:

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# 1. Carregar os dados (ajuste os nomes das colunas conforme sua base)
colunas = ['user_id', 'game', 'behavior', 'value', 'zero']
df = pd.read_csv('/root/.cache/kagglehub/datasets/tamber/steam-video-games/versions/3/steam-200k.csv',names=colunas)


In [42]:
df

,user_id,game,behavior,value,zero
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0
2,151603712,Fallout 4,purchase,1.0,0
3,151603712,Fallout 4,play,87.0,0
4,151603712,Spore,purchase,1.0,0
...,...,...,...,...,...
199995,128470551,Titan Souls,play,1.5,0
199996,128470551,Grand Theft Auto Vice City,purchase,1.0,0
199997,128470551,Grand Theft Auto Vice City,play,1.5,0
199998,128470551,RUSH,purchase,1.0,0


In [49]:
df_play = df[df['behavior'] == 'play']

In [50]:
import numpy as np
user_matrix_game = df_play.pivot_table(index='user_id', columns='game', values='value', fill_value=0)

In [54]:
from sklearn.preprocessing import MinMaxScaler
#usando a func log1p para suavizar os dados gigantes.
user_matrix_game_log  = np.log1p(user_matrix_game)

scaler = MinMaxScaler()
dados_normalizados = scaler.fit_transform(user_matrix_game_log)

from sklearn.cluster import KMeans
modelo_k = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = modelo_k.fit_predict(dados_normalizados)


In [56]:
print(user_matrix_game[user_matrix_game['cluster'] == 0].drop('cluster', axis=1).mean().sort_values(ascending=False).head(5))

game
Counter-Strike Global Offensive    211.278450
Dota 2                             202.972881
Team Fortress 2                    137.623729
The Elder Scrolls V Skyrim          87.040436
Counter-Strike Source               60.414528
dtype: float64


In [57]:
# Comparando com o próximo grupo
print("Top 5 do Cluster 1:")
print(user_matrix_game[user_matrix_game['cluster'] == 1].drop('cluster', axis=1).mean().sort_values(ascending=False).head(5))

Top 5 do Cluster 1:
game
Dota 2                             510.368197
Counter-Strike Global Offensive     42.350631
Counter-Strike                      15.711711
Team Fortress 2                      5.770551
Counter-Strike Source                4.283238
dtype: float64


In [61]:
print("Top 5 do Cluster 2:")
print(user_matrix_game[user_matrix_game['cluster'] == 2 ].drop('cluster', axis=1).mean().sort_values(ascending=False).head(5))

Top 5 do Cluster 2:
game
Counter-Strike Global Offensive    17.593386
Team Fortress 2                    11.612335
Counter-Strike                     11.082247
Sid Meier's Civilization V          8.649494
Counter-Strike Source               6.924747
dtype: float64


In [62]:
print("Top 5 do Cluster 3:")
print(user_matrix_game[user_matrix_game['cluster'] == 3 ].drop('cluster', axis=1).mean().sort_values(ascending=False).head(5))

Top 5 do Cluster 3:
game
Left 4 Dead                   243.0
Borderlands 2                 162.0
The Elder Scrolls V Skyrim    157.0
Trove                         111.0
APB Reloaded                   78.0
dtype: float64


In [63]:
print("Tamanho de cada grupo de jogadores:")
print(user_matrix_game['cluster'].value_counts())

Tamanho de cada grupo de jogadores:
cluster
2    9193
1    1742
0     413
3       1
4       1
Name: count, dtype: int64


In [87]:
def recomendar_jogos(user_id):

    cluster_do_user = user_matrix_game.loc[user_id, 'cluster']


    recomendados = user_matrix_game[user_matrix_game['cluster'] == cluster_do_user].mean().sort_values(ascending=False).head(10)

    return recomendados.index.tolist()
print(f"Recomendações para o usuário: {recomendar_jogos(144736)}")


Recomendações para o usuário: ['Counter-Strike Global Offensive', 'Team Fortress 2', 'Counter-Strike', "Sid Meier's Civilization V", 'Counter-Strike Source', 'The Elder Scrolls V Skyrim', 'Football Manager 2013', 'Call of Duty Modern Warfare 2 - Multiplayer', 'Football Manager 2012', 'Football Manager 2014']


In [94]:
user_matrix_game.index

Index([     5250,     76767,     86540,    144736,    181212,    229911,
          298950,    381543,    547685,    554278,
       ...
       309228590, 309255941, 309262440, 309265377, 309404240, 309434439,
       309554670, 309626088, 309824202, 309903146],
      dtype='int64', name='user_id', length=11350)